# Exercises

There are three exercises in this notebook:

1. Use the cross-validation method to test the linear regression with different $\alpha$ values, at least three.
2. Implement a SGD method that will train the Lasso regression for 10 epochs.
3. Extend the Fisher's classifier to work with two features. Use the class as the $y$.

## 1. Cross-validation linear regression

You need to change the variable ``alpha`` to be a list of alphas. Next do a loop and finally compare the results.

In [1]:
import numpy as np # dodane tak o

x = np.array([188, 181, 197, 168, 167, 187, 178, 194, 140, 176, 168, 192, 173, 142, 176]).reshape(-1, 1).reshape(15,1)
y = np.array([141, 106, 149, 59, 79, 136, 65, 136, 52, 87, 115, 140, 82, 69, 121]).reshape(-1, 1).reshape(15,1)

x = np.asmatrix(np.c_[np.ones((15,1)),x])

I = np.identity(2)
alphas = [0.5, 0.3, 0.1, 0.01] # change here

# add 1-3 line of code here
def ridge(alphas):
    results = []
    for alpha in alphas:
        w = np.linalg.inv(x.T*x + alpha * I)*x.T*y
        w = np.asarray(w).ravel() # Tutaj wcześniej było samo .ravel() ale teraz jest np.asarray(w).ravel() aby uniknąć błędu związane z typem danych
        results.append(w)
    return zip(alphas, results)

# add 1-3 lines to compare the results
for a, w in ridge(alphas):
    mse = np.mean(np.square(y - x @ w.reshape(-1, 1)))
    print(f"alpha={a}: w0={w[0]:.4f}, w1={w[1]:.4f}, MSE={mse:.4f}")

alpha=0.5: w0=-36.9752, w1=0.8032, MSE=549.7711
alpha=0.3: w0=-54.2370, w1=0.9010, MSE=509.7668
alpha=0.1: w0=-101.7240, w1=1.1698, MSE=426.0451
alpha=0.01: w0=-167.8553, w1=1.5442, MSE=373.7938


## 2. Implement based on the Ridge regression example, the Lasso regression.

Please implement the SGD method and compare the results with the sklearn Lasso regression results. 

In [7]:
from sklearn.linear_model import Lasso

def sgd(alpha):
    X = np.asarray(x, dtype=float)
    y_vec = np.asarray(y, dtype=float).ravel()

    n_samples, n_features = X.shape
    w_sgd = np.zeros(n_features)

    lr = 1e-5
    epochs = 10

    for _ in range(epochs):
        for i in np.random.permutation(n_samples):
            pred = X[i] @ w_sgd
            err = pred - y_vec[i]
            grad = err * X[i]

            l1_grad = alpha * np.sign(w_sgd)
            l1_grad[0] = 0.0

            w_sgd -= lr * (grad + l1_grad)

    lasso = Lasso(alpha=alpha, fit_intercept=False, max_iter=100000, tol=1e-8)
    lasso.fit(X, y_vec)

    mse_sgd = np.mean(np.square(y_vec - X @ w_sgd))
    mse_sklearn = np.mean(np.square(y_vec - X @ lasso.coef_))

    return w_sgd, lasso.coef_, mse_sgd, mse_sklearn

In [9]:
x = np.array([188, 181, 197, 168, 167, 187, 178, 194, 140, 176, 168, 192, 173, 142, 176]).reshape(-1, 1).reshape(15,1)
y = np.array([141, 106, 149, 59, 79, 136, 65, 136, 52, 87, 115, 140, 82, 69, 121]).reshape(-1, 1).reshape(15,1)

x = np.asmatrix(np.c_[np.ones((15,1)),x])

I = np.identity(2)
alpha = [0.5, 0.3, 0.1, 0.01]


for alpha in alphas:
    w_sgd, w_sklearn, mse_sgd, mse_sklearn = sgd(alpha)
    print(f"alpha={alpha}: SGD weights={w_sgd}, MSE={mse_sgd:.4f}; sklearn weights={w_sklearn}, MSE={mse_sklearn:.4f}")
    print("-" * 50)

alpha=0.5: SGD weights=[0.00119553 0.62598295], MSE=684.4464; sklearn weights=[-122.20378211    1.2857087 ], MSE=401.8577
--------------------------------------------------
alpha=0.3: SGD weights=[0.00097512 0.59735102], MSE=653.0015; sklearn weights=[-145.69179255    1.41868173], MSE=382.9608
--------------------------------------------------
alpha=0.1: SGD weights=[0.0008547  0.62431196], MSE=681.2128; sklearn weights=[-169.17980435    1.55165477], MSE=373.5124
--------------------------------------------------
alpha=0.01: SGD weights=[0.0014322  0.69012627], MSE=938.9991; sklearn weights=[-179.7494089     1.61149264], MSE=372.3431
--------------------------------------------------


## 3. Extend the Fisher's classifier

Please extend the targets of the ``iris_data`` variable and use it as the $y$.

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

iris_data = load_iris()
iris_df = pd.DataFrame(iris_data.data,columns=iris_data.feature_names)
iris_df.head()

x = iris_df[['sepal width (cm)','sepal length (cm)']].values.reshape(-1, 1).reshape(150, 2)
y = iris_data.target.astype(float).reshape(-1, 1)

dataset_size = np.size(x)
mean_x, mean_y = np.mean(x), np.mean(y)

SS_xy = np.sum(y * x) - dataset_size * mean_y * mean_x
SS_xx = np.sum(x * x) - dataset_size * mean_x * mean_x

a = SS_xy / SS_xx
b = mean_y - a * mean_x


y_pred = a * x + b